# Industry-Style Agentic RAG with LangGraph

## Why this is industry-style

A simple RAG system searches only the private knowledge base.

An industry-style Agentic RAG system should do this:

```text
Question
  ↓
Search private knowledge base first
  ↓
Grade the private evidence
  ↓
If private evidence is good → answer from KB
  ↓
If private evidence is weak → search the web
  ↓
Grade web evidence
  ↓
Generate a grounded answer with source type
```

This design is useful because private documents may be incomplete or outdated.

## 1. Install Dependencies

## 2. Configure API Keys

In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except EnvironmentError:
    raise

_GROQ_API_KEY = os.getenv("GROQ_API_KEY")
_GROQ_MODEL = os.getenv("GROQ_MODEL")
_TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
_PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
_GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
_LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
_HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
_EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME")
_PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")
_PINECONE_INDEX_NAMESPACE = os.getenv("PINECONE_INDEX_NAMESPACE")
missing_api_key_list = []
if not _GROQ_API_KEY:
    missing_api_key_list.append("GROQ_API_KEY")
if not _GROQ_MODEL:
    missing_api_key_list.append("GROQ_MODEL")
if not _TAVILY_API_KEY:
    missing_api_key_list.append("TAVILY_API_KEY")
if not _PINECONE_API_KEY:
    missing_api_key_list.append("PINECONE_API_KEY")
if not _LANGSMITH_API_KEY:
    missing_api_key_list.append("LANGSMITH_API_KEY")
if not _GEMINI_API_KEY:
    missing_api_key_list.append("GEMINI_API_KEY")
if not _HUGGINGFACE_API_KEY:
    missing_api_key_list.append("HUGGINGFACE_API_KEY")
if not _EMBEDDING_MODEL_NAME:
    missing_api_key_list.append("EMBEDDING_MODEL_NAME")
if not _PINECONE_INDEX_NAME:
    missing_api_key_list.append("PINECONE_INDEX_NAME")
if not _PINECONE_INDEX_NAMESPACE:
    missing_api_key_list.append("PINECONE_INDEX_NAMESPACE")

if missing_api_key_list is None:
    print("missing_api_key_list: ", missing_api_key_list)
else:
    print("No missing_api_keys")

No missing_api_keys


In [3]:
print(_EMBEDDING_MODEL_NAME)

sentence-transformers/all-MiniLM-L6-v2


In [4]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# making the pinecone object
pinecone_object= Pinecone(api_key=_PINECONE_API_KEY)

# # loading pinecone index
index_name = pinecone_object.Index(_PINECONE_INDEX_NAME)

# connecting the vector store using pinecone 
vector_store= PineconeVectorStore(
    index=index_name,
    embedding=_EMBEDDING_MODEL_NAME,
    namespace=_PINECONE_INDEX_NAMESPACE
)

# making retriver
retriever= vector_store.as_retriever(
    search_kwargs={"k":5,'namespace':_PINECONE_INDEX_NAMESPACE}
)

print("Existing Index: ",_PINECONE_INDEX_NAME)

Existing Index:  it-support-agentic-rag-indexname


## 3. Load Documents

create private knowledge base from a real web document:

`https://docs.langchain.com/oss/python/langgraph/agentic-rag`


The web fallback is separate. That means:

- First source: indexed LangGraph documentation page/private knowledge base
- Fallback source: live web search through Tavily

In [5]:
from langchain_community.document_loaders import WebBaseLoader
url= "https://docs.langchain.com/langsmith/python/managed-deep-agents-overview"
loader= WebBaseLoader(
    web_path=url,
)
raw_docs=loader.load()
print(raw_docs)
print("Length of data:",len(raw_docs))
print("source: ",raw_docs[0].metadata.get("source"))
print("\nPreview:")
print(raw_docs[0].page_content[:1500])

C:\Users\sudhakar\AppData\Local\Temp\ipykernel_12436\1451838045.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://docs.langchain.com/langsmith/python/managed-deep-agents-overview', 'title': 'Managed Deep Agents - Docs by LangChain', 'description': 'Build your agent as a directory of files while LangSmith runs the harness and runtime.', 'language': 'en'}, page_content='\n\n\n\n\nManaged Deep Agents - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedManaged Deep AgentsOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonGet startedBETAOverviewQuickstartProject structureMove from Deep AgentsAgent capabilitiesAgent definitionInstructionsSkillsToolsBuilt-in searchConnectionsMCPMiddlewareMemorySandboxesBrowser useSubagentsIdentityChannelsSchedulesEvalsRunti

## 4. Split the document into chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
)

chunks= splitter.split_documents(raw_docs)
print("No of chunks: ",len(chunks))
print("\nChunk Overview:\n")
print(chunks[0].page_content[:900])

No of chunks:  13

Chunk Overview:

Managed Deep Agents - Docs by LangChain


## 5.Create Free Local Embeddings

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model= HuggingFaceEmbeddings(
    model_name=_EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)
sample_vector= embedding_model.embed_query("What is AI")
print(sample_vector)
print("dimention of sample_vector: ", len(sample_vector))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[-0.028200605884194374, -0.017675522714853287, 0.011596136726439, 0.01601574569940567, 0.045705925673246384, -0.016152145341038704, 0.06501340121030807, 0.031805988401174545, 0.029278932139277458, 0.05825493484735489, -0.042929694056510925, -0.003123966511338949, 0.009426412172615528, -0.04228344187140465, -0.044077642261981964, 0.07152608036994934, -0.024430735036730766, -0.04126305133104324, -0.07005303353071213, -0.0744963139295578, -0.013285133987665176, 0.00889817625284195, -0.05688982084393501, -0.07097277790307999, -0.03665563091635704, 0.09129757434129715, 0.028841977939009666, -0.07827780395746231, -0.010502156801521778, -0.01822674460709095, 0.01755453273653984, -0.06115095689892769, 0.08683191984891891, 0.015249386429786682, -0.040124934166669846, 0.02645130082964897, -0.04383980855345726, -0.04861941933631897, 0.0956171303987503, -0.02186714857816696, -0.014999916777014732, -0.040611714124679565, 0.01712803728878498, -0.0696665421128273, 0.10470209270715714, 0.1244667321443

In [8]:
def embedding_model_dimention(embedding_model):
    sample_vector= embedding_model.embed_query("What is AI?")
    dimention= len(sample_vector)
    return dimention


print("embedding_model_dimention: ",embedding_model_dimention(embedding_model))

embedding_model_dimention:  384


## 6. creating the pinecone vector database

The embedding model used in this notebook produces **384-dimensional vectors**, so the Pinecone index must use dimension `384`.

# steps:
# 1. create pinecone vector store or index using Pinecone class
# 2. send the documents to pinecone index using lagchain_pinecone class and making vector store
# 3. make the retriever

In [9]:
from pinecone import Pinecone,ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

# 1. create pinecone vector store or index using Pinecone class
pinecone_object= Pinecone(
    api_key=_PINECONE_API_KEY,
)
# creating the pinecone index if it doesn't exist
existing_indexes= [index_info["name"] for index_info in pinecone_object.list_indexes()]
print(existing_indexes)

if _PINECONE_INDEX_NAME not in existing_indexes:
    pinecone_object.create_index(
        name=_PINECONE_INDEX_NAME,
        dimension=embedding_model_dimention(embedding_model),
        metric='cosine',
        spec=ServerlessSpec(cloud='aws',region='us-east-1')
    )
# wait until the index is prepared in pinecone
while not pinecone_object.describe_index(name=_PINECONE_INDEX_NAME).status['ready']:
    time.wait(1)
print("pinecone index is ready: ",_PINECONE_INDEX_NAME)    

# 2. send the documents to pinecone index using lagchain_pinecone class and making vector store
vector_store= PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    index_name=_PINECONE_INDEX_NAME,
    namespace=_PINECONE_INDEX_NAMESPACE
)

# 3. make the retriever
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace":_PINECONE_INDEX_NAMESPACE
    }
)
print('Pinecone vector database and its retriever is ready.')

['it-support-agentic-rag-indexname']
pinecone index is ready:  it-support-agentic-rag-indexname
Pinecone vector database and its retriever is ready.


# 7.Loading the existing index

In [10]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# making the pinecone object
pinecone_object= Pinecone(api_key=_PINECONE_API_KEY)

# # loading pinecone index
index_name = pinecone_object.Index(_PINECONE_INDEX_NAME)

# connecting the vector store using pinecone 
vector_store= PineconeVectorStore(
    index=index_name,
    embedding=embedding_model,
    namespace=_PINECONE_INDEX_NAMESPACE
)

# making retriver
retriever= vector_store.as_retriever(
    search_kwargs={"k":5,'namespace':_PINECONE_INDEX_NAMESPACE}
)

print("Existing Index: ",_PINECONE_INDEX_NAME)

Existing Index:  it-support-agentic-rag-indexname


# 8. testing the private KB

In [11]:
query="what is langchain?"

kb_docs= retriever.invoke(query)

for i, doc in enumerate(kb_docs):
    print(f"\n----KB DOCS{i+1}---")
    print("source: ", doc.metadata.get('source'))
    print("content: ", doc.page_content[:500])



----KB DOCS1---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS2---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS3---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS4---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS5---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain


# 8. initializing groq LLM

In [21]:
from langchain_groq import ChatGroq
llm= ChatGroq(
    model=_GROQ_MODEL,
    temperature=0
)
query= llm.invoke("What is langchain in 10 words?")
print(query.content)

LangChain is a framework for building AI‑powered language applications efficiently.


# 9. Initiate Tavily websearch

Tavily will be used only when the private knowledge base is insufficient.

This is important because we do not want to send every private/company question to the public web.

In [13]:
from langchain_tavily import TavilySearch

web_search= TavilySearch(
    max_results= 5,
    topic= 'general',
    include_answer= True,
    # include_raw_content=True,
    # include_images=True
)

print("Tavily web_search tool is ready")

Tavily web_search tool is ready


In [14]:
web_search.invoke("recent AI news?")

{'query': 'recent AI news?',
 'follow_up_questions': None,
 'answer': "Recent AI news includes Google's stock hitting a six-month low due to concerns over its AI search summaries affecting ad revenue, and AI tools being used by police forces to combat art theft. Additionally, there is mention of ethical issues surrounding AI, including generative AI and large language models, as well as the use of AI in predictive analytics and speech recognition. The sources do not provide a definitive answer to specific questions about particular events or figures.",
 'images': [],
 'results': [{'url': 'https://techcrunch.com/category/artificial-intelligence',
   'title': 'AI News & Artificial Intelligence | TechCrunch',
   'content': '# AI\n\nNews coverage on artificial intelligence and machine learning tech, the companies building them, and the ethical issues AI raises today. This encompasses generative AI, including large language models, text-to-image and text-to-video models; speech recognition 

# 10. Defining structured Decisions and Langgraph state

We need reliable yes/no or route outputs from the LLM.

Structured outputs help avoid parsing messy natural language.

In [19]:
from typing import List, Literal, TypedDict
from pydantic import BaseModel,Field
from langchain_core.documents import Document

class DecisionRoute(BaseModel):
    route: Literal["Kb",'direct']= Field(
        description="Using 'kb' when we need to answer using Agentic RAG; Use 'direct' for simple questions where 'kb' is not required."
    )

class GradingAnswer(BaseModel):
    grading: Literal['good','weak']= Field(
        description="grade the answer as 'good' if the evidence is enough to answer the query, otherwise return 'weak'."
    )

class AgentState(TypedDict):
    query: str
    source_used:str
    current_query: str

    kb_docs: List[Document]
    web_results:str

    kb_grade:str
    web_grade:str
    answer: str   

    retry_count:int 

# Nodes

## 11. Node 1

The router decides:

- `kb`: use private knowledge base
- `direct`: answer without retrieval

In a real company assistant, this step may also route between many sources:
HR docs, finance docs, product docs, SQL database, web search, etc.

In [22]:
router_llm= llm.with_structured_output(DecisionRoute,method='json_mode')

def route_question(state:AgentState):
    query= state["query"]

    decision= router_llm.invoke(f'''
    you are a router for Agentic Rag assistant.

    route to 'kb' if the user asks about:
    - Agentic RAG
    - LangGraph Agentic RAG workflow
    - retrieval grading
    - query rewriting
    - RAG architecture
    - retriever tools
    - web fallback in RAG

    route to 'direct' only for greetings, thanks or very simple conversations.

    query:
    {query}

    Return response as a valid json:
    Example:
    {{'route':'kb'}} 
    ''')

    return {
        'query':query,
        'source_used': decision.route
    }

def route_after_routing(state: AgentState)->Literal['retriever_kb','direct_answer']:
    if state['source_used']=='kb':
        return 'retriever_kb'
    return 'direct_answer'

# 12. Node-2 : Retrive from the private KB

In [23]:
def retriever_kb(state:AgentState)->dict:
    query= state["current_query"]
    chunks= retriever.invoke(input=query)

    print(f"[kb_retriever] query:{query}")
    print(f"[kb_retriever] retrieved:{len(chunks)} chunks")

    return {
        "kb_docs":chunks,
    }

# 13. Node 3: Grade Private KB Evidence

This is the key step that decides whether we can answer from the private KB or need web fallback.

In [ ]:
kb_grader_llm = llm.with_structured_output(
    schema=GradingAnswer, method='json_mode')

def grade_kb_evidence(state:AgentState):
    query= state["query"]
    context= ''.join(
        f"source: {doc.metadata.get('source')} \n {doc.page_content}"
        for doc in {state['kb_docs']}
    )

    grade= kb_grader_llm.invoke(f""" 
        you are an evidence grader.
        query: {query},
        private kb evidence: {context}

        if the private kb evidence can answer the query, return "good".
        Return "weak" if it can not answer the query or is incomplete.

        Return response in valid JSON only.
        Example:
            {{"grade":"weak}}
        """)

    print(f"[kb_grader] : {grade.grade}")    
    return {"kb_grade":grade.grade}

def decide_after_kb_grade(state: AgentState):
    if state["kb_grade"]=="good":
        return "generate_from_kb"
    return "search_web"

ValueError: Unrecognized method argument. Expected one of 'function_calling', 'json_mode', or 'json_schema'. Received: 'json-mode'